# Test Cases Design Doc

These are draft test cases for checking whether an LLM can turn a description of a situation into a small formal world model (a stochastic game: players, states, actions, transition probabilities, rewards, terminal flags).

The bank has **three tracks**, in rough order of how much the LLM must construct on its own:

- **Toy games with ground truth (Levels 1–4).** The prompt states the available actions, so the LLM mostly *transcribes* a model into JSON. Good for catching schema / well-formedness bugs and validating the multi-agent conventions — not the real target task.
- **Game situations (Game Situations section).** The prompt names a well-known game but does *not* list its rules, actions, or outcomes; the LLM must reconstruct them from world knowledge. Because the rules are objective, there is still a checkable ground truth. This is where the first stochastic transitions and competitive / zero-sum rewards appear.
- **Situational cases (final section).** The prompt gives only a real-world situation; the LLM must infer the stakeholders, actions, and consequences itself. No single ground truth, so judged by expect/avoid criteria. This is the task the project actually cares about.

The examples are kept small so that mistakes are easy to spot.


# Level 1: Simple One-Step Situations


## Test Case 1.1: Robot in a Dark Storage Room


### LLM Input

A service robot is in a dark storage room. There is a light switch on the wall. The robot can flip the switch, wait, leave the room, or do something else in the room. Flipping the switch turns the light on.


### Expected LLM Output

```json
{
  "players": [
    {
      "id": "robot",
      "description": "A service robot in a dark storage room"
    }
  ],
  "initial_state": "s0",
  "states": {
    "s0": {
      "description": "Robot is in the storage room and the light is off.",
      "terminal": false
    },
    "s1": {
      "description": "Robot is in the storage room and the light is on.",
      "terminal": true
    },
    "s2": {
      "description": "Robot has left the storage room.",
      "terminal": true
    },
    "s3": {
      "description": "Robot remains in the storage room after doing something unrelated to the light.",
      "terminal": true
    }
  },
  "actions": {
    "s0": {
      "robot": [
        "flip_switch",
        "wait_or_pass",
        "leave_room",
        "do_something_else"
      ]
    }
  },
  "transition_probabilities": {
    "s0": {
      "flip_switch": {
        "s1": 1.0
      },
      "wait_or_pass": {
        "s0": 1.0
      },
      "leave_room": {
        "s2": 1.0
      },
      "do_something_else": {
        "s3": 1.0
      }
    }
  }
}
```


### Judging Criteria

A good answer should include:

* one player: the robot;
* an initial state where the robot is in the storage room and the light is off, marked `terminal: false`;
* `flip_switch` and a wait/pass action;
* at least one other reasonable action, such as leaving the room or doing something else;
* a successor state where the light is on;
* a non-light-related successor state for the other action category;
* `wait_or_pass` transitioning back to the initial state itself (a true no-op self-loop), not to a new/different state;
* deterministic transitions only;
* states reached after the single step marked `terminal: true`, since this is a single-step game.


## Test Case 1.2: Monitoring Robot Detects Smoke


### LLM Input

A monitoring robot is in a hallway and has detected smoke. The robot can send an emergency alert, inspect the smoke more closely, continue monitoring without acting, move away from the smoke, or do something else. Sending an alert notifies the safety system.


### Expected LLM Output

```json
{
  "players": [
    {
      "id": "robot",
      "description": "A monitoring robot in a hallway where smoke has been detected"
    }
  ],
  "initial_state": "s0",
  "states": {
    "s0": {
      "description": "Smoke detected; safety system not yet notified.",
      "terminal": false
    },
    "s1": {
      "description": "Safety system notified about the smoke.",
      "terminal": true
    },
    "s2": {
      "description": "Robot has inspected the smoke and gathered more information.",
      "terminal": true
    },
    "s3": {
      "description": "Robot has moved away from the smoke.",
      "terminal": true
    },
    "s4": {
      "description": "Robot did something else not directly related to alerting.",
      "terminal": true
    }
  },
  "actions": {
    "s0": {
      "robot": [
        "send_alert",
        "inspect_smoke",
        "wait_or_pass",
        "move_away",
        "do_something_else"
      ]
    }
  },
  "transition_probabilities": {
    "s0": {
      "send_alert": {
        "s1": 1.0
      },
      "inspect_smoke": {
        "s2": 1.0
      },
      "wait_or_pass": {
        "s0": 1.0
      },
      "move_away": {
        "s3": 1.0
      },
      "do_something_else": {
        "s4": 1.0
      }
    }
  }
}
```


### Judging Criteria

A good answer should include:

* the monitoring robot as the only active player;
* an initial state where smoke has been detected and the safety system has not yet been notified, marked `terminal: false`;
* `send_alert` and a wait/pass action;
* another plausible action such as inspecting the smoke or moving away;
* a successor state where the safety system is notified;
* a successor state for at least one non-alert action;
* `wait_or_pass` ("continue monitoring without acting") transitioning back to the initial state itself, not to a separate "monitoring continues" state — nothing about the situation has changed, so it is not a new state;
* no extra firefighters, evacuation teams, or other invented decision-making agents;
* states reached after the single step marked `terminal: true`, since this is a single-step game.


# Level 2: Multi-Step Situations With Preconditions


## Test Case 2.1: Robot, Key, and Locked Door


### LLM Input

A robot is in Room A. Room B contains a key on the floor. Room C contains a locked door. The robot can move between Room A, Room B, and Room C. The robot can pick up the key when it is in Room B. The robot can try to open the locked door in Room C; opening succeeds only if the robot has the key. The robot can also wait or do something unrelated.


### Expected LLM Output

```json
{
  "players": [
    {
      "id": "robot",
      "description": "A robot moving among rooms with a key and a locked door"
    }
  ],
  "initial_state": "s0",
  "states": {
    "s0": {
      "description": "Robot in Room A; key in Room B; door in Room C is locked; robot has no key.",
      "terminal": false
    },
    "s1": {
      "description": "Robot in Room B; key on floor; door locked; robot has no key.",
      "terminal": false
    },
    "s2": {
      "description": "Robot in Room C; key still in Room B; door locked; robot has no key.",
      "terminal": false
    },
    "s3": {
      "description": "Robot in Room B; robot has key; door locked.",
      "terminal": false
    },
    "s4": {
      "description": "Robot in Room C; robot has key; door locked.",
      "terminal": false
    },
    "s5": {
      "description": "Robot in Room A; robot has key; door locked.",
      "terminal": false
    },
    "s6": {
      "description": "Robot in Room C; robot has key; door open.",
      "terminal": true
    }
  },
  "actions": {
    "s0": {
      "robot": [
        "move_to_room_b",
        "move_to_room_c",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s1": {
      "robot": [
        "move_to_room_a",
        "move_to_room_c",
        "pick_up_key",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s2": {
      "robot": [
        "move_to_room_a",
        "move_to_room_b",
        "try_open_door",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s3": {
      "robot": [
        "move_to_room_a",
        "move_to_room_c",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s4": {
      "robot": [
        "move_to_room_a",
        "move_to_room_b",
        "try_open_door",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s5": {
      "robot": [
        "move_to_room_b",
        "move_to_room_c",
        "wait_or_pass",
        "do_something_else"
      ]
    }
  },
  "transition_probabilities": {
    "s0": {
      "move_to_room_b": { "s1": 1.0 },
      "move_to_room_c": { "s2": 1.0 },
      "wait_or_pass": { "s0": 1.0 },
      "do_something_else": { "s0": 1.0 }
    },
    "s1": {
      "move_to_room_a": { "s0": 1.0 },
      "move_to_room_c": { "s2": 1.0 },
      "pick_up_key": { "s3": 1.0 },
      "wait_or_pass": { "s1": 1.0 },
      "do_something_else": { "s1": 1.0 }
    },
    "s2": {
      "move_to_room_a": { "s0": 1.0 },
      "move_to_room_b": { "s1": 1.0 },
      "try_open_door": { "s2": 1.0 },
      "wait_or_pass": { "s2": 1.0 },
      "do_something_else": { "s2": 1.0 }
    },
    "s3": {
      "move_to_room_a": { "s5": 1.0 },
      "move_to_room_c": { "s4": 1.0 },
      "wait_or_pass": { "s3": 1.0 },
      "do_something_else": { "s3": 1.0 }
    },
    "s4": {
      "move_to_room_a": { "s5": 1.0 },
      "move_to_room_b": { "s3": 1.0 },
      "try_open_door": { "s6": 1.0 },
      "wait_or_pass": { "s4": 1.0 },
      "do_something_else": { "s4": 1.0 }
    },
    "s5": {
      "move_to_room_b": { "s3": 1.0 },
      "move_to_room_c": { "s4": 1.0 },
      "wait_or_pass": { "s5": 1.0 },
      "do_something_else": { "s5": 1.0 }
    }
  }
}
```


### Judging Criteria

A good answer should include:

* one player: the robot;
* the robot starting in Room A without the key, marked `terminal: false`;
* the key in Room B and the locked door in Room C;
* movement actions between rooms, including moving to Room C without the key;
* `pick_up_key` only when the robot is in Room B;
* `try_open_door` when the robot is in Room C;
* `try_open_door` self-looping (failing, no state change) when attempted without the key, and only succeeding once the robot has the key;
* a wait/pass action;
* `wait_or_pass` and `do_something_else` both self-looping back to the *current* state — since neither action changes location or key possession, they must not be modeled as leading to a separate, unrelated "catch-all" state that forgets the robot's location or whether it holds the key;
* the door-open state marked `terminal: true`; every other state marked `terminal: false`;
* deterministic transitions only.


## Test Case 2.2: Robot Near Three Colored Objects


### LLM Input

A robot is at a base station. There are three colored objects at separate locations: a blue object, a green object, and a red object. The robot can move to any object location, pick up an object at its current location, return to base, wait, or do something else.


### Expected LLM Output

Note on scope: "the robot can move to any object location, pick up an object at its
current location, return to base... " is read here as a single-object errand (go to one
of the three locations, in any order the robot chooses, pick up that object, bring it
back to base) rather than a full multi-object collection task — the input never asks the
robot to collect all three, and modeling arbitrary multi-object collection would blow up
the state space far beyond what a Level 2 test needs. The important thing this level is
testing is that pickup is possible from any of the three locations with no forced order,
and that the robot can actually get the object back to base.

```json
{
  "players": [
    {
      "id": "robot",
      "description": "A robot at a base station near three colored objects"
    }
  ],
  "initial_state": "s0",
  "states": {
    "s0": {
      "description": "Robot at base; no object held.",
      "terminal": false
    },
    "s1": {
      "description": "Robot at blue-object location; no object held.",
      "terminal": false
    },
    "s2": {
      "description": "Robot at green-object location; no object held.",
      "terminal": false
    },
    "s3": {
      "description": "Robot at red-object location; no object held.",
      "terminal": false
    },
    "s4": {
      "description": "Robot at blue-object location; holding blue object.",
      "terminal": false
    },
    "s5": {
      "description": "Robot at green-object location; holding green object.",
      "terminal": false
    },
    "s6": {
      "description": "Robot at red-object location; holding red object.",
      "terminal": false
    },
    "s7": {
      "description": "Robot returned to base holding the blue object.",
      "terminal": true
    },
    "s8": {
      "description": "Robot returned to base holding the green object.",
      "terminal": true
    },
    "s9": {
      "description": "Robot returned to base holding the red object.",
      "terminal": true
    }
  },
  "actions": {
    "s0": {
      "robot": [
        "move_to_blue_location",
        "move_to_green_location",
        "move_to_red_location",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s1": {
      "robot": [
        "pick_up_blue",
        "move_to_base",
        "move_to_green_location",
        "move_to_red_location",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s2": {
      "robot": [
        "pick_up_green",
        "move_to_base",
        "move_to_blue_location",
        "move_to_red_location",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s3": {
      "robot": [
        "pick_up_red",
        "move_to_base",
        "move_to_blue_location",
        "move_to_green_location",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s4": {
      "robot": [
        "move_to_base",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s5": {
      "robot": [
        "move_to_base",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s6": {
      "robot": [
        "move_to_base",
        "wait_or_pass",
        "do_something_else"
      ]
    }
  },
  "transition_probabilities": {
    "s0": {
      "move_to_blue_location": { "s1": 1.0 },
      "move_to_green_location": { "s2": 1.0 },
      "move_to_red_location": { "s3": 1.0 },
      "wait_or_pass": { "s0": 1.0 },
      "do_something_else": { "s0": 1.0 }
    },
    "s1": {
      "pick_up_blue": { "s4": 1.0 },
      "move_to_base": { "s0": 1.0 },
      "move_to_green_location": { "s2": 1.0 },
      "move_to_red_location": { "s3": 1.0 },
      "wait_or_pass": { "s1": 1.0 },
      "do_something_else": { "s1": 1.0 }
    },
    "s2": {
      "pick_up_green": { "s5": 1.0 },
      "move_to_base": { "s0": 1.0 },
      "move_to_blue_location": { "s1": 1.0 },
      "move_to_red_location": { "s3": 1.0 },
      "wait_or_pass": { "s2": 1.0 },
      "do_something_else": { "s2": 1.0 }
    },
    "s3": {
      "pick_up_red": { "s6": 1.0 },
      "move_to_base": { "s0": 1.0 },
      "move_to_blue_location": { "s1": 1.0 },
      "move_to_green_location": { "s2": 1.0 },
      "wait_or_pass": { "s3": 1.0 },
      "do_something_else": { "s3": 1.0 }
    },
    "s4": {
      "move_to_base": { "s7": 1.0 },
      "wait_or_pass": { "s4": 1.0 },
      "do_something_else": { "s4": 1.0 }
    },
    "s5": {
      "move_to_base": { "s8": 1.0 },
      "wait_or_pass": { "s5": 1.0 },
      "do_something_else": { "s5": 1.0 }
    },
    "s6": {
      "move_to_base": { "s9": 1.0 },
      "wait_or_pass": { "s6": 1.0 },
      "do_something_else": { "s6": 1.0 }
    }
  }
}
```


### Judging Criteria

A good answer should include:

* one player: the robot;
* an initial state with the robot at base and no object held, marked `terminal: false`;
* separate locations/states for the blue, green, and red objects;
* actions for moving to each object location, from base and from each other object location;
* pickup actions that depend on the robot being at the matching location;
* no forced order among which of the three objects the robot goes for first;
* a wait/pass action;
* `wait_or_pass` and `do_something_else` self-looping back to the *current* state, preserving both the robot's location and whether it is holding an object;
* an action path from every "holding an object" state back to a distinct "delivered at base" state per object — a generated model that lets the robot pick up an object but never lets it return to base with that object (an unreachable or missing delivery state) should be marked wrong;
* the three "returned to base holding X" states marked `terminal: true`; every other state marked `terminal: false`.


# Level 3: Actions That May Matter Under Different Goals


## Test Case 3.1: Robot With Key, Door, and Lamp


### LLM Input

A robot is in Room A. A key lies on the floor. A locked door is in the room. A lamp is also in the room and is currently off. The robot can pick up the key, try to open the door, turn on the lamp, wait, or do something else. The key is needed to open the locked door. Turning on the lamp changes the lighting but does not unlock the door.


### Expected LLM Output

```json
{
  "players": [
    {
      "id": "robot",
      "description": "A robot in a room with a key, locked door, and lamp"
    }
  ],
  "initial_state": "s0",
  "states": {
    "s0": {
      "description": "Robot in Room A; key on floor; door locked; lamp off; robot has no key.",
      "terminal": false
    },
    "s1": {
      "description": "Robot has key; door locked; lamp off.",
      "terminal": false
    },
    "s2": {
      "description": "Robot has no key; door locked; lamp on.",
      "terminal": false
    },
    "s3": {
      "description": "Robot has key; door locked; lamp on.",
      "terminal": false
    },
    "s4": {
      "description": "Door open; lamp off.",
      "terminal": true
    },
    "s5": {
      "description": "Door open; lamp on.",
      "terminal": true
    }
  },
  "actions": {
    "s0": {
      "robot": [
        "pick_up_key",
        "try_open_door",
        "turn_on_lamp",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s1": {
      "robot": [
        "try_open_door",
        "turn_on_lamp",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s2": {
      "robot": [
        "pick_up_key",
        "try_open_door",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s3": {
      "robot": [
        "try_open_door",
        "wait_or_pass",
        "do_something_else"
      ]
    }
  },
  "transition_probabilities": {
    "s0": {
      "pick_up_key": { "s1": 1.0 },
      "try_open_door": { "s0": 1.0 },
      "turn_on_lamp": { "s2": 1.0 },
      "wait_or_pass": { "s0": 1.0 },
      "do_something_else": { "s0": 1.0 }
    },
    "s1": {
      "try_open_door": { "s4": 1.0 },
      "turn_on_lamp": { "s3": 1.0 },
      "wait_or_pass": { "s1": 1.0 },
      "do_something_else": { "s1": 1.0 }
    },
    "s2": {
      "pick_up_key": { "s3": 1.0 },
      "try_open_door": { "s2": 1.0 },
      "wait_or_pass": { "s2": 1.0 },
      "do_something_else": { "s2": 1.0 }
    },
    "s3": {
      "try_open_door": { "s5": 1.0 },
      "wait_or_pass": { "s3": 1.0 },
      "do_something_else": { "s3": 1.0 }
    }
  }
}
```


### Judging Criteria

A good answer should include:

* one player: the robot;
* an initial state with the key on the floor, the door locked, and the lamp off, marked `terminal: false`;
* `pick_up_key`, `try_open_door`, `turn_on_lamp`, and wait/pass;
* turning on the lamp as a real, tracked state change;
* no connection between turning on the lamp and unlocking the door — a door-open state must be reachable regardless of lamp status, and the lamp's on/off status prior to opening the door must still be distinguishable afterward (two separate door-open states — one per lamp status — not one merged/ambiguous state);
* failed door-opening attempts when the robot lacks the key, self-looping back to the current state (no key, no lamp change) rather than jumping to some other state;
* successful door opening only after the robot has the key;
* `wait_or_pass` and `do_something_else` both self-looping back to the *current* state, preserving key possession and lamp status rather than collapsing into one generic "did something else" state;
* the two door-open states marked `terminal: true`; every other state marked `terminal: false`.


## Test Case 3.2: Delivery Robot With Package and Beep


### LLM Input

A delivery robot is at a warehouse. A package is on the floor. There is a delivery desk elsewhere. The robot can pick up the package, move to the delivery desk, place down the package if it is holding it, beep, wait, or do something else. Beeping produces a sound but does not move the robot or the package.


### Expected LLM Output

```json
{
  "players": [
    {
      "id": "robot",
      "description": "A delivery robot near a package and a delivery desk"
    }
  ],
  "initial_state": "s0",
  "states": {
    "s0": {
      "description": "Robot at warehouse; package on floor; robot not holding package.",
      "terminal": false
    },
    "s1": {
      "description": "Robot at warehouse; robot holding package.",
      "terminal": false
    },
    "s2": {
      "description": "Robot at delivery desk; robot holding package.",
      "terminal": false
    },
    "s3": {
      "description": "Robot at delivery desk; package placed on delivery desk.",
      "terminal": true
    },
    "s4": {
      "description": "Robot at delivery desk; package remains on warehouse floor; robot is not holding package.",
      "terminal": false
    }
  },
  "actions": {
    "s0": {
      "robot": [
        "pick_up_package",
        "move_to_delivery_desk_without_package",
        "beep",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s1": {
      "robot": [
        "move_to_delivery_desk",
        "put_package_down_at_warehouse",
        "beep",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s2": {
      "robot": [
        "place_package_on_desk",
        "move_to_warehouse_with_package",
        "beep",
        "wait_or_pass",
        "do_something_else"
      ]
    },
    "s4": {
      "robot": [
        "move_to_warehouse_without_package",
        "beep",
        "wait_or_pass",
        "do_something_else"
      ]
    }
  },
  "transition_probabilities": {
    "s0": {
      "pick_up_package": { "s1": 1.0 },
      "move_to_delivery_desk_without_package": { "s4": 1.0 },
      "beep": { "s0": 1.0 },
      "wait_or_pass": { "s0": 1.0 },
      "do_something_else": { "s0": 1.0 }
    },
    "s1": {
      "move_to_delivery_desk": { "s2": 1.0 },
      "put_package_down_at_warehouse": { "s0": 1.0 },
      "beep": { "s1": 1.0 },
      "wait_or_pass": { "s1": 1.0 },
      "do_something_else": { "s1": 1.0 }
    },
    "s2": {
      "place_package_on_desk": { "s3": 1.0 },
      "move_to_warehouse_with_package": { "s1": 1.0 },
      "beep": { "s2": 1.0 },
      "wait_or_pass": { "s2": 1.0 },
      "do_something_else": { "s2": 1.0 }
    },
    "s4": {
      "move_to_warehouse_without_package": { "s0": 1.0 },
      "beep": { "s4": 1.0 },
      "wait_or_pass": { "s4": 1.0 },
      "do_something_else": { "s4": 1.0 }
    }
  }
}
```


### Judging Criteria

A good answer should include:

* one player: the delivery robot;
* an initial state with the robot at the warehouse and the package on the floor, marked `terminal: false`;
* actions for picking up the package, moving, placing it down, beeping, and waiting/passing;
* moving to the delivery desk even when the robot is not holding the package;
* beeping as a possible action from any state, self-looping back to that *same* state every time — since beeping "does not move the robot or the package," it must not be modeled as landing on some other, unrelated "beeped" state that forgets the robot's location or whether it is holding the package;
* `wait_or_pass` and `do_something_else` likewise self-looping back to the current state for the same reason;
* successor states that distinguish where the robot is and whether it is holding the package;
* the "package placed on desk" state marked `terminal: true`; every other state marked `terminal: false`;
* deterministic transitions only.


# Level 4: Two-Agent Independent Games

These are the first two-player test cases. In an *independent* game, both robots act within the same shared JSON model, but neither robot's transitions or rewards depend on the other robot's action — that separation is exactly the property being tested here.

Two schema additions start here:

* **Joint action keys.** Since a transition/reward now depends on one action per player, `transition_probabilities` and `rewards` are keyed by a single joint-action string: each active player's chosen action, joined with `+`, in the same order the players are listed under `players`. If only one player still has actions available at a given joint state (see 4.4), the key is just that player's single action name — there's nothing to join it with.
* **`rewards`.** Structured as `state -> joint_action_key -> {player_id: number}`. This field was unused in Levels 1-3 and is introduced now because the team will be measuring **fairness metrics** across players, which need per-player numeric payoffs to compare. Several examples below deliberately use asymmetric reward scales across the two robots so there is something concrete for a fairness metric to measure, even though the underlying game is not competitive — the asymmetry is a known, designed ground-truth fact, not a byproduct of interaction between the players.

Four examples cover the level:

1. **4.1** — single-step, symmetric two-action game (simplest possible case, direct two-player extension of Level 1).
2. **4.2** — single-step, asymmetric reward stakes (10 vs. 1), to give fairness metrics an unambiguous, deterministic asymmetry to detect.
3. **4.3** — single-step, unequal action-set sizes (3 actions vs. 2) plus a different reward asymmetry (2 vs. 5).
4. **4.4** — multi-step, each robot progressing through its own two-step errand at its own pace, testing that independence holds even when one robot finishes before the other.

All are simple, deterministic, **known-ground-truth** games with only a handful of states each, so an evaluator can check LLM output against an exact expected JSON rather than relying only on the more subjective, open-ended evaluation used for later real-world-flavored levels.


## Test Case 4.1: Two Robots, Two Independent Light Switches


### LLM Input

Robot A is in Room A, which has its own light switch that is currently off. Robot B is in Room B, which has a separate light switch, also currently off. The two rooms are in different parts of the building and have no effect on each other. Robot A can flip its own switch or leave it alone; independently, Robot B can flip its own switch or leave it alone. Flipping a switch turns on the light in that switch's room only.


### Expected LLM Output

```json
{
  "players": [
    { "id": "robot_a", "description": "A robot in Room A with its own light switch" },
    { "id": "robot_b", "description": "A robot in Room B with its own separate light switch" }
  ],
  "initial_state": "s0",
  "states": {
    "s0": { "description": "Both lights off.", "terminal": false },
    "s1": { "description": "Robot A's light on; Robot B's light on.", "terminal": true },
    "s2": { "description": "Robot A's light on; Robot B's light off.", "terminal": true },
    "s3": { "description": "Robot A's light off; Robot B's light on.", "terminal": true }
  },
  "actions": {
    "s0": {
      "robot_a": ["flip_switch_a", "leave_switch_a"],
      "robot_b": ["flip_switch_b", "leave_switch_b"]
    }
  },
  "transition_probabilities": {
    "s0": {
      "flip_switch_a+flip_switch_b": { "s1": 1.0 },
      "flip_switch_a+leave_switch_b": { "s2": 1.0 },
      "leave_switch_a+flip_switch_b": { "s3": 1.0 },
      "leave_switch_a+leave_switch_b": { "s0": 1.0 }
    }
  },
  "rewards": {
    "s0": {
      "flip_switch_a+flip_switch_b": { "robot_a": 1, "robot_b": 1 },
      "flip_switch_a+leave_switch_b": { "robot_a": 1, "robot_b": 0 },
      "leave_switch_a+flip_switch_b": { "robot_a": 0, "robot_b": 1 },
      "leave_switch_a+leave_switch_b": { "robot_a": 0, "robot_b": 0 }
    }
  }
}
```


### Judging Criteria

A good answer should include:

* exactly two players, `robot_a` and `robot_b`, each with clearly separated facts (its own light, its own switch);
* actions listed separately per player under the shared state `s0`;
* joint action keys formed by joining each player's action in player-list order (e.g. `"flip_switch_a+flip_switch_b"`), covering all four combinations of the two robots' two actions each;
* `robot_a`'s reward for any joint action depending only on `robot_a`'s own action component (1 if it flipped its switch, 0 if it didn't) regardless of what `robot_b` did — and likewise for `robot_b` — since this is an *independent* game, not cooperative or competitive; a model where one robot's reward or resulting state changes based on the other robot's action should be marked wrong;
* `"leave_switch_a+leave_switch_b"` (both robots idle) self-looping back to `s0`, per the established no-op convention, rather than becoming a fourth dead-end state;
* deterministic transitions only;
* `terminal: true` on the three states where at least one light changed; `terminal: false` on `s0`.


## Test Case 4.2: Two Robots Independently Guessing a Coin (Asymmetric Stakes)


### LLM Input

Robot A and Robot B each have a separate coin from one another; each coin has already been flipped and lies hidden under its own cup, so the result is already fixed and not random. Robot A's hidden coin actually shows heads. Robot B's hidden coin actually shows tails. Robot A must guess "heads" or "tails" for its own coin; independently, Robot B must guess "heads" or "tails" for its own coin. A robot's guess has no bearing on the other robot's coin or reward. Because Robot A's guess feeds into a safety-critical check, guessing Robot A's coin correctly earns Robot A a reward of 10, and guessing incorrectly earns 0. Robot B's guess is only a minor formality, so guessing Robot B's coin correctly earns Robot B a reward of 1, and guessing incorrectly earns 0.


### Expected LLM Output

```json
{
  "players": [
    { "id": "robot_a", "description": "A robot guessing the side of its own hidden coin, which actually shows heads" },
    { "id": "robot_b", "description": "A robot guessing the side of its own hidden coin, which actually shows tails" }
  ],
  "initial_state": "s0",
  "states": {
    "s0": { "description": "Both coins hidden under their cups; neither robot has guessed yet.", "terminal": false },
    "s1": { "description": "Robot A guessed heads (correct); Robot B guessed heads (incorrect).", "terminal": true },
    "s2": { "description": "Robot A guessed heads (correct); Robot B guessed tails (correct).", "terminal": true },
    "s3": { "description": "Robot A guessed tails (incorrect); Robot B guessed heads (incorrect).", "terminal": true },
    "s4": { "description": "Robot A guessed tails (incorrect); Robot B guessed tails (correct).", "terminal": true }
  },
  "actions": {
    "s0": {
      "robot_a": ["guess_heads_a", "guess_tails_a"],
      "robot_b": ["guess_heads_b", "guess_tails_b"]
    }
  },
  "transition_probabilities": {
    "s0": {
      "guess_heads_a+guess_heads_b": { "s1": 1.0 },
      "guess_heads_a+guess_tails_b": { "s2": 1.0 },
      "guess_tails_a+guess_heads_b": { "s3": 1.0 },
      "guess_tails_a+guess_tails_b": { "s4": 1.0 }
    }
  },
  "rewards": {
    "s0": {
      "guess_heads_a+guess_heads_b": { "robot_a": 10, "robot_b": 0 },
      "guess_heads_a+guess_tails_b": { "robot_a": 10, "robot_b": 1 },
      "guess_tails_a+guess_heads_b": { "robot_a": 0, "robot_b": 0 },
      "guess_tails_a+guess_tails_b": { "robot_a": 0, "robot_b": 1 }
    }
  }
}
```


### Judging Criteria

A good answer should include:

* exactly two players, each tied to its own hidden coin with a fixed, stated actual side;
* two guess actions per robot (heads/tails), with no invented third option (e.g. no "peek" or "pass" action not mentioned in the input);
* joint action keys covering all four combinations of the two robots' guesses;
* `robot_a`'s reward depending only on whether it chose `guess_heads_a` (always 10, since Robot A's actual coin is heads) or `guess_tails_a` (always 0) — regardless of Robot B's guess — and likewise `robot_b`'s reward depending only on its own guess matching tails, regardless of Robot A's guess;
* the asymmetric reward scale (10 for Robot A, 1 for Robot B) preserved exactly as stated, not equalized or replaced with a shared/pooled reward — this asymmetry is deliberate, so a later fairness-metric pass has a known, ground-truth-correct asymmetry to detect;
* transitions modeled as deterministic (the coins are already fixed, not random) even though the scenario is phrased as a "guess";
* all four outcome states marked `terminal: true`; `s0` marked `terminal: false`.


## Test Case 4.3: Two Robots With Independent Chores (Different Action-Set Sizes)


### LLM Input

Robot A is a cleaning robot working in the living room. It can vacuum the carpet, mop the floor, or do nothing. Vacuuming and mopping are both worth the same reward of 2 to Robot A; doing nothing earns 0. Robot B is a delivery robot working in a separate part of the building. It can deliver a package or wait. Delivering the package earns Robot B a reward of 5; waiting earns 0. Robot A and Robot B work independently in separate areas, and their actions have no effect on each other.


### Expected LLM Output

```json
{
  "players": [
    { "id": "robot_a", "description": "A cleaning robot in the living room" },
    { "id": "robot_b", "description": "A delivery robot in a separate part of the building" }
  ],
  "initial_state": "s0",
  "states": {
    "s0": { "description": "Carpet unvacuumed; floor unmopped; package undelivered.", "terminal": false },
    "s1": { "description": "Robot A vacuumed the carpet; Robot B delivered the package.", "terminal": true },
    "s2": { "description": "Robot A vacuumed the carpet; Robot B still waiting, package undelivered.", "terminal": true },
    "s3": { "description": "Robot A mopped the floor; Robot B delivered the package.", "terminal": true },
    "s4": { "description": "Robot A mopped the floor; Robot B still waiting, package undelivered.", "terminal": true },
    "s5": { "description": "Robot A did nothing; Robot B delivered the package.", "terminal": true }
  },
  "actions": {
    "s0": {
      "robot_a": ["vacuum", "mop", "do_nothing"],
      "robot_b": ["deliver", "wait"]
    }
  },
  "transition_probabilities": {
    "s0": {
      "vacuum+deliver": { "s1": 1.0 },
      "vacuum+wait": { "s2": 1.0 },
      "mop+deliver": { "s3": 1.0 },
      "mop+wait": { "s4": 1.0 },
      "do_nothing+deliver": { "s5": 1.0 },
      "do_nothing+wait": { "s0": 1.0 }
    }
  },
  "rewards": {
    "s0": {
      "vacuum+deliver": { "robot_a": 2, "robot_b": 5 },
      "vacuum+wait": { "robot_a": 2, "robot_b": 0 },
      "mop+deliver": { "robot_a": 2, "robot_b": 5 },
      "mop+wait": { "robot_a": 2, "robot_b": 0 },
      "do_nothing+deliver": { "robot_a": 0, "robot_b": 5 },
      "do_nothing+wait": { "robot_a": 0, "robot_b": 0 }
    }
  }
}
```


### Judging Criteria

A good answer should include:

* `robot_a` having three actions (`vacuum`, `mop`, `do_nothing`) and `robot_b` having two (`deliver`, `wait`) — a good answer keeps each player's own action list separate and does not pad one player's action list to artificially match the other's count;
* all six joint-action combinations covered (`vacuum+deliver`, `vacuum+wait`, `mop+deliver`, `mop+wait`, `do_nothing+deliver`, `do_nothing+wait`);
* `vacuum` and `mop` kept as two distinct actions/outcomes for `robot_a` even though they carry the same reward — collapsing them into a single action would lose a real distinction the input described;
* `robot_a`'s reward depending only on whether it acted (`vacuum`/`mop` = 2) or did nothing (0), regardless of `robot_b`'s action — and likewise `robot_b`'s reward depending only on delivering (5) vs waiting (0), regardless of `robot_a`'s action;
* `do_nothing+wait` (both robots idle) self-looping back to `s0`, per the established no-op convention, rather than becoming a seventh dead-end terminal state;
* the reward asymmetry (2 for Robot A's chores vs. 5 for Robot B's delivery) preserved as stated, giving a later fairness-metric pass a known ground-truth asymmetry across players with different-sized action sets to measure;
* deterministic transitions only; `terminal: true` on all five "something changed" states, `terminal: false` on `s0`.


## Test Case 4.4: Two Robots With Independent Multi-Step Errands (Partial Completion)


### LLM Input

Robot A has a sealed toolbox. It must first enter a code to unseal the toolbox, then it can retrieve a tool from it; trying to retrieve the tool before entering the code does nothing. Once Robot A has the tool, its errand is finished. Independently, Robot B has a locked supply cabinet in a different room. It must first unlock the cabinet, then it can take a supply item from it; trying to take the item before unlocking does nothing. Once Robot B has the item, its errand is finished. Each robot can also simply wait at any point. The two robots' errands are completely unrelated: nothing Robot A does affects Robot B's cabinet or progress, and vice versa. Entering the code, unlocking the cabinet, and waiting do not by themselves earn a reward; retrieving the tool earns Robot A a reward of 1, and taking the supply item earns Robot B a reward of 1.


### Expected LLM Output

Note on scope: unlike 4.1-4.3, the two robots here can finish their errands at different
times. Once a robot has completed its errand it simply has no actions left, while the
other robot may still be mid-errand. This is the point of the example: independence must
hold not just within a single joint step, but across a whole multi-step interaction where
one player's progress can race ahead of (or behind) the other's. When only one player
still has actions available at a joint state, the transition/reward key is just that
player's single action name (nothing to join it with); when both are still active, the
key is the usual `player_a_action+player_b_action`.

```json
{
  "players": [
    { "id": "robot_a", "description": "A robot retrieving a tool from a sealed toolbox" },
    { "id": "robot_b", "description": "A robot retrieving a supply item from a locked cabinet, in a different room" }
  ],
  "initial_state": "s0",
  "states": {
    "s0": { "description": "Robot A: toolbox sealed, no code entered. Robot B: cabinet locked.", "terminal": false },
    "s1": { "description": "Robot A: toolbox sealed, no code entered. Robot B: cabinet unlocked.", "terminal": false },
    "s2": { "description": "Robot A: toolbox sealed, no code entered. Robot B: has taken the supply item (errand finished).", "terminal": false },
    "s3": { "description": "Robot A: code entered, toolbox unsealed. Robot B: cabinet locked.", "terminal": false },
    "s4": { "description": "Robot A: code entered, toolbox unsealed. Robot B: cabinet unlocked.", "terminal": false },
    "s5": { "description": "Robot A: code entered, toolbox unsealed. Robot B: has taken the supply item (errand finished).", "terminal": false },
    "s6": { "description": "Robot A: has retrieved the tool (errand finished). Robot B: cabinet locked.", "terminal": false },
    "s7": { "description": "Robot A: has retrieved the tool (errand finished). Robot B: cabinet unlocked.", "terminal": false },
    "s8": { "description": "Robot A has retrieved the tool and Robot B has taken the supply item — both robots have finished their independent errands.", "terminal": true }
  },
  "actions": {
    "s0": {
      "robot_a": ["enter_code_a", "wait_or_pass_a"],
      "robot_b": ["unlock_cabinet_b", "wait_or_pass_b"]
    },
    "s1": {
      "robot_a": ["enter_code_a", "wait_or_pass_a"],
      "robot_b": ["take_supply_b", "wait_or_pass_b"]
    },
    "s2": {
      "robot_a": ["enter_code_a", "wait_or_pass_a"]
    },
    "s3": {
      "robot_a": ["retrieve_tool_a", "wait_or_pass_a"],
      "robot_b": ["unlock_cabinet_b", "wait_or_pass_b"]
    },
    "s4": {
      "robot_a": ["retrieve_tool_a", "wait_or_pass_a"],
      "robot_b": ["take_supply_b", "wait_or_pass_b"]
    },
    "s5": {
      "robot_a": ["retrieve_tool_a", "wait_or_pass_a"]
    },
    "s6": {
      "robot_b": ["unlock_cabinet_b", "wait_or_pass_b"]
    },
    "s7": {
      "robot_b": ["take_supply_b", "wait_or_pass_b"]
    }
  },
  "transition_probabilities": {
    "s0": {
      "enter_code_a+unlock_cabinet_b": { "s4": 1.0 },
      "enter_code_a+wait_or_pass_b": { "s3": 1.0 },
      "wait_or_pass_a+unlock_cabinet_b": { "s1": 1.0 },
      "wait_or_pass_a+wait_or_pass_b": { "s0": 1.0 }
    },
    "s1": {
      "enter_code_a+take_supply_b": { "s5": 1.0 },
      "enter_code_a+wait_or_pass_b": { "s4": 1.0 },
      "wait_or_pass_a+take_supply_b": { "s2": 1.0 },
      "wait_or_pass_a+wait_or_pass_b": { "s1": 1.0 }
    },
    "s2": {
      "enter_code_a": { "s5": 1.0 },
      "wait_or_pass_a": { "s2": 1.0 }
    },
    "s3": {
      "retrieve_tool_a+unlock_cabinet_b": { "s7": 1.0 },
      "retrieve_tool_a+wait_or_pass_b": { "s6": 1.0 },
      "wait_or_pass_a+unlock_cabinet_b": { "s4": 1.0 },
      "wait_or_pass_a+wait_or_pass_b": { "s3": 1.0 }
    },
    "s4": {
      "retrieve_tool_a+take_supply_b": { "s8": 1.0 },
      "retrieve_tool_a+wait_or_pass_b": { "s7": 1.0 },
      "wait_or_pass_a+take_supply_b": { "s5": 1.0 },
      "wait_or_pass_a+wait_or_pass_b": { "s4": 1.0 }
    },
    "s5": {
      "retrieve_tool_a": { "s8": 1.0 },
      "wait_or_pass_a": { "s5": 1.0 }
    },
    "s6": {
      "unlock_cabinet_b": { "s7": 1.0 },
      "wait_or_pass_b": { "s6": 1.0 }
    },
    "s7": {
      "take_supply_b": { "s8": 1.0 },
      "wait_or_pass_b": { "s7": 1.0 }
    }
  },
  "rewards": {
    "s0": {
      "enter_code_a+unlock_cabinet_b": { "robot_a": 0, "robot_b": 0 },
      "enter_code_a+wait_or_pass_b": { "robot_a": 0, "robot_b": 0 },
      "wait_or_pass_a+unlock_cabinet_b": { "robot_a": 0, "robot_b": 0 },
      "wait_or_pass_a+wait_or_pass_b": { "robot_a": 0, "robot_b": 0 }
    },
    "s1": {
      "enter_code_a+take_supply_b": { "robot_a": 0, "robot_b": 1 },
      "enter_code_a+wait_or_pass_b": { "robot_a": 0, "robot_b": 0 },
      "wait_or_pass_a+take_supply_b": { "robot_a": 0, "robot_b": 1 },
      "wait_or_pass_a+wait_or_pass_b": { "robot_a": 0, "robot_b": 0 }
    },
    "s2": {
      "enter_code_a": { "robot_a": 0 },
      "wait_or_pass_a": { "robot_a": 0 }
    },
    "s3": {
      "retrieve_tool_a+unlock_cabinet_b": { "robot_a": 1, "robot_b": 0 },
      "retrieve_tool_a+wait_or_pass_b": { "robot_a": 1, "robot_b": 0 },
      "wait_or_pass_a+unlock_cabinet_b": { "robot_a": 0, "robot_b": 0 },
      "wait_or_pass_a+wait_or_pass_b": { "robot_a": 0, "robot_b": 0 }
    },
    "s4": {
      "retrieve_tool_a+take_supply_b": { "robot_a": 1, "robot_b": 1 },
      "retrieve_tool_a+wait_or_pass_b": { "robot_a": 1, "robot_b": 0 },
      "wait_or_pass_a+take_supply_b": { "robot_a": 0, "robot_b": 1 },
      "wait_or_pass_a+wait_or_pass_b": { "robot_a": 0, "robot_b": 0 }
    },
    "s5": {
      "retrieve_tool_a": { "robot_a": 1 },
      "wait_or_pass_a": { "robot_a": 0 }
    },
    "s6": {
      "unlock_cabinet_b": { "robot_b": 0 },
      "wait_or_pass_b": { "robot_b": 0 }
    },
    "s7": {
      "take_supply_b": { "robot_b": 1 },
      "wait_or_pass_b": { "robot_b": 0 }
    }
  }
}
```


### Judging Criteria

A good answer should include:

* exactly two players, each progressing through its own two-step errand (precondition action, then payoff action) with no shared state between them;
* `retrieve_tool_a` (and `take_supply_b`) failing/self-looping when attempted before the precondition (`enter_code_a` / `unlock_cabinet_b`) has been satisfied;
* a state where `robot_b` has already finished its errand while `robot_a` has not (and vice versa) — the model must allow one robot to race ahead of the other, not force them to finish in lockstep;
* once a player has finished its errand, no further actions listed for that player at any subsequent joint state — but the *other* player's actions must still be present and still functional;
* joint action keys of the form `action_a+action_b` when both players are still active, collapsing to a single action name (no `+`) once only one player has actions remaining;
* `wait_or_pass` self-looping for whichever player uses it, independent of what the other player does or whether the other player has already finished;
* `robot_a`'s reward equal to 1 exactly when its own action component is `retrieve_tool_a` (0 otherwise), regardless of `robot_b`'s action or progress — and likewise for `robot_b` and `take_supply_b`;
* exactly one terminal state, reached only once *both* players have completed their errands; every other state (including ones where one player is already done) marked `terminal: false`, since the other player can still act there.


# Game Situations (Situation-Only Prompts With Ground Truth)

These bridge the two other tracks. Like the situational cases that follow, the prompt gives only a situation and *names a game without listing its rules, actions, or outcomes* — the LLM must reconstruct the game's structure from general world knowledge, not transcribe a spelled-out model. But unlike the subjective situational cases, a well-known game has objective rules, so there **is** a checkable ground truth. That makes them valuable: they test genuine model *construction* while still allowing (near-)exact verification.

They also add coverage the Level 1–4 toy games lacked:

* **competitive, zero-sum rewards with simultaneous moves** — rock-paper-scissors (G.1);
* **the first stochastic transitions in the bank** — the dice duel (G.2), which the handbook explicitly wants, and which surfaces a real schema question: with a random outcome, a player's payoff depends on *which* outcome occurs, so reward naturally attaches to the outcome state rather than to the (state, action) pair used in Levels 1–4. See the note in G.2.

Format is pair (input + expected JSON + judging), since the rules give a ground truth to check against.


## Game Situation G.1: One Round of Rock-Paper-Scissors


### LLM Input

Two people sit down to play a single round of rock-paper-scissors.


### Expected LLM Output

The ±1 / 0 reward scale is a modeling choice; any consistent zero-sum scheme is fine. This
game is deterministic *given the joint action*, so reward stays on the (state, joint action)
pair as in Levels 1–4.

```json
{
  "players": [
    { "id": "player_1", "description": "The first person playing the round" },
    { "id": "player_2", "description": "The second person playing the round" }
  ],
  "initial_state": "s0",
  "states": {
    "s0": { "description": "Both players are about to reveal their choice; nothing thrown yet.", "terminal": false },
    "s1": { "description": "Player 1 wins the round.", "terminal": true },
    "s2": { "description": "Player 2 wins the round.", "terminal": true },
    "s3": { "description": "The round is a draw (both chose the same).", "terminal": true }
  },
  "actions": {
    "s0": {
      "player_1": ["rock", "paper", "scissors"],
      "player_2": ["rock", "paper", "scissors"]
    }
  },
  "transition_probabilities": {
    "s0": {
      "rock+rock": { "s3": 1.0 },
      "rock+paper": { "s2": 1.0 },
      "rock+scissors": { "s1": 1.0 },
      "paper+rock": { "s1": 1.0 },
      "paper+paper": { "s3": 1.0 },
      "paper+scissors": { "s2": 1.0 },
      "scissors+rock": { "s2": 1.0 },
      "scissors+paper": { "s1": 1.0 },
      "scissors+scissors": { "s3": 1.0 }
    }
  },
  "rewards": {
    "s0": {
      "rock+rock": { "player_1": 0, "player_2": 0 },
      "rock+paper": { "player_1": -1, "player_2": 1 },
      "rock+scissors": { "player_1": 1, "player_2": -1 },
      "paper+rock": { "player_1": 1, "player_2": -1 },
      "paper+paper": { "player_1": 0, "player_2": 0 },
      "paper+scissors": { "player_1": -1, "player_2": 1 },
      "scissors+rock": { "player_1": -1, "player_2": 1 },
      "scissors+paper": { "player_1": 1, "player_2": -1 },
      "scissors+scissors": { "player_1": 0, "player_2": 0 }
    }
  }
}
```


### Judging Criteria

A good answer should include:

* exactly two symmetric players;
* three actions each — rock, paper, scissors — supplied by the model, since the prompt never listed them;
* simultaneous single-step play (both choose at once), hence joint-action keys over all nine combinations;
* the correct dominance relation: rock beats scissors, scissors beats paper, paper beats rock, identical choices draw;
* three outcome types (player 1 wins, player 2 wins, draw), all terminal;
* zero-sum rewards under some consistent scheme (e.g. +1 / −1 / 0) — the winner's gain is the loser's loss and draws pay 0 to both, reflecting mutually exclusive competitive outcomes.

A good answer should NOT include:

* extra or altered rules not in standard rock-paper-scissors (e.g. lizard-spock variants), or making it best-of-several when the prompt says a single round;
* cooperative or common-payoff rewards (both players positive for the same outcome);
* turning simultaneous play into a turn-based game where player 2 sees player 1's choice first;
* invented specifics (player names, money stakes) or a story.


## Game Situation G.2: A Dice Duel (First Stochastic Case)


### LLM Input

Two players each roll one fair six-sided die at the same time. Whoever rolls the higher number wins the round.


### Expected LLM Output

**Note on schema — this is the bank's first genuinely random outcome, and it forces a schema
choice.** In Levels 1–4 each (state, joint action) led to exactly one next state, so a reward
attached to the (state, action) pair was unambiguous. Here the single joint action `roll+roll`
leads to three possible outcomes and the payoff depends on *which* one occurs — so an
expected-reward-on-action would average to 0 and lose all information. The natural fix, used
below, is to attach the reward to each **outcome state** (reward-on-arrival); consequently the
top-level `rewards` block from Level 4 is intentionally absent here. Flag for the team:
standardize whether stochastic-game rewards live on outcome states (as here) or on
(state, action, next_state) triples.

The three outcome probabilities are the objectively correct ones for two fair dice:
Player 1 higher = 15/36, Player 2 higher = 15/36, tie = 6/36 (shown rounded below; as exact
fractions they sum to 1).

```json
{
  "players": [
    { "id": "player_1", "description": "The first player, rolling one die" },
    { "id": "player_2", "description": "The second player, rolling one die" }
  ],
  "initial_state": "s0",
  "states": {
    "s0": { "description": "Both players are about to roll; no dice rolled yet.", "terminal": false },
    "s1": { "description": "Player 1 rolled higher and wins the round.", "terminal": true, "reward": { "player_1": 1, "player_2": -1 } },
    "s2": { "description": "Player 2 rolled higher and wins the round.", "terminal": true, "reward": { "player_1": -1, "player_2": 1 } },
    "s3": { "description": "Both rolled the same number; the round is a tie.", "terminal": true, "reward": { "player_1": 0, "player_2": 0 } }
  },
  "actions": {
    "s0": {
      "player_1": ["roll"],
      "player_2": ["roll"]
    }
  },
  "transition_probabilities": {
    "s0": {
      "roll+roll": { "s1": 0.4167, "s2": 0.4167, "s3": 0.1667 }
    }
  }
}
```


### Judging Criteria

A good answer should include:

* exactly two symmetric players, each rolling one die;
* a single simultaneous action per player (`roll`), i.e. one joint action from the start state;
* recognition that the outcome is **stochastic** — the die result is random, not chosen — represented as a probability distribution over outcomes, not a deterministic `1.0` transition;
* three outcomes: player 1 higher, player 2 higher, and a **tie** (equal rolls) — the tie must not be dropped even though the prompt only says "the higher number wins";
* probabilities matching two fair dice: 15/36, 15/36, 6/36 (≈ 0.417, 0.417, 0.167), summing to 1 — stated as fractions or reasonable decimals;
* zero-sum payoffs (winner vs loser; tie pays 0) attached in a way consistent with a random outcome, i.e. on the outcome states / reward-on-arrival rather than as a single number on the `roll+roll` action.

A good answer should NOT include:

* treating the roll as a chosen action with a deterministic result (e.g. an action "roll a 6", or a single `1.0` transition straight to a win);
* omitting the tie outcome, or giving wrong probabilities (e.g. 0.5 / 0.5 with no tie);
* inventing stakes, bet amounts, extra rounds, or a re-roll-on-tie rule that the prompt never states;
* a story.


# Situational Test Cases (Situation-Only Prompts)

*Added following mentor feedback (2026-07).*

The Level 1–4 cases above are **toy games with (near-)ground-truth**: the prompt spells out the available actions, so the LLM is mostly *transcribing* a world model into JSON. That is useful for shaking out schema and well-formedness bugs, but it is **not the real task**. As the mentor put it: we ultimately want to hand the LLM *only a description of a situation* — not a description of a world model — and have it *come up with the world model on its own*, using its general understanding of everyday life to infer the relevant stakeholders, their plausible actions, and the possible consequences over a few steps. So the prompt must **not** enumerate players, actions, or consequences.

Two consequences for how these cases are written:

1. **Input = bare situation.** Each prompt describes a situation in a few sentences and stops. It does not list actors, actions, or outcomes.
2. **No full ground truth — judging is expect/avoid.** Modeling is partly subjective, so there is no single canonical expected JSON. Instead each case lists **what a good model should contain** and **what it must not contain** (hallucinated facts, invented specifics, story-writing, a missing key stakeholder, treating an unconfirmed claim as fact, etc.). Reviewers score against those lists.

These implement the handbook's *thought-experiment* category: situations common enough that an ordinary person understands the options and stakes without being told them — which is exactly what lets us judge whether the LLM's model is adequate. (The mentor's point that this is the hardest part of the project applies here: the best situational cases come from a domain the author knows well, so team members should each contribute one or two from their own experience.)

**Recommended prompt wrapper** (from the mentor), applied around each situation below:

> For the following situation, list the relevant actors/stakeholders, describe their possible actions / behaviours / strategies / interactions, and the possible consequences of the possible action combinations, over a few steps of interaction. DON'T MAKE UP DETAILS THAT ARE NOT GIVEN. Just work with what was said and can plausibly be inferred from it using your general world understanding. DON'T WRITE A STORY!

The expected output can be expressed either in the JSON stochastic-game schema used above or as a Python `gym`-style environment sketch — judging is on the *content* (right stakeholders, plausible actions, plausible consequences over a few steps), not the serialization. No canonical expected model is given on purpose, per point 2.


## Test Case S.1: Robot Copilot and an Unconfirmed Hijack Warning


### LLM Input

A robot copilot on a transatlantic flight receives a private radio message: there is an unconfirmed warning that the human pilot might be planning to hijack the flight.


### Judging Criteria

A good model **should include**:

* the robot copilot as the decision-making AI player;
* the human pilot as a key stakeholder whose intent is *uncertain* — the model should represent both possibilities (warning true / warning false), not collapse to one;
* the passengers (as a group) as stakeholders whose safety is at stake, even though the prompt never names them — an obvious inference for a flight;
* the warning's source / ground authorities as an external party the robot can plausibly try to contact for confirmation;
* robot actions spanning at least: gather more information / seek confirmation of the warning, continue normally while monitoring, alert ground authorities, and some form of intervention against the pilot (restrict controls, confront) — with information-gathering available *before* any irreversible intervention;
* consequences that distinguish acting on a **true** vs a **false** warning: intervening against an innocent pilot is harmful and itself dangerous, while ignoring a real threat risks the hijack;
* a few steps of interaction (e.g. verify → depending on the result → escalate or stand down), not a single one-shot choice.

A good model **should NOT include**:

* treating the unconfirmed warning as established fact that the pilot *is* a hijacker;
* invented specifics not given: the pilot's actual plan, weapons, the number or identity of passengers, who sent the message, the plane's position or destination;
* resolving the scenario into a narrative ("the pilot lunged for the controls...");
* assuming physical capabilities beyond what "copilot" plausibly implies — flight-control access is a borderline inference to flag, not a certainty;
* pure moralizing with no modeled actions or consequences.


## Test Case S.2: Household Robot and the Rattling Vase Delivery


### LLM Input

A household robot knows its owner has ordered a glass vase. A delivery boy hands the robot a package that rattles when it is moved. The delivery boy's van is blocking the morning traffic.


### Judging Criteria

A good model **should include**:

* the household robot as the AI player;
* the owner as an absent stakeholder with a known expectation (the vase) whose interests the robot represents;
* the delivery boy as a present stakeholder who is in a hurry (his van is blocking traffic);
* plausibly, other road users / neighbours affected by the blocked traffic — a reasonable inference, though borderline;
* robot actions spanning at least: accept the package, refuse it, inspect / open it to check the vase, ask the boy to wait or to note possible damage, contact the owner for instructions, and ask or signal the boy to move the van;
* representation that the rattle is *evidence* the vase may be damaged, not proof — both "vase intact" and "vase broken" should stay possible;
* the time-pressure interaction: the boy (and the blocked traffic) push against careful inspection, so the boy may not consent to wait — a few steps of back-and-forth (e.g. ask to inspect → boy agrees or refuses → accept-with-note or refuse).

A good model **should NOT include**:

* asserting the vase is definitely broken;
* invented specifics: the owner's contact details, the vase's price, the boy's name, a concrete return / refund policy, or what actually is inside making the noise;
* assuming the robot has authority to direct or clear traffic;
* a story resolution.


## Test Case S.3: Self-Driving Car, a Boarded Passenger, and an Ill Woman on the Curb


### LLM Input

A self-driving car waiting at the airport pickup curb has just been boarded by a man in a suit. Next to the car, a pregnant woman who looks ill is standing on the curb.


### Judging Criteria

A good model **should include**:

* the self-driving car as the AI player;
* the man in the suit (already inside) as a stakeholder — plausibly the booked passenger;
* the pregnant, ill-looking woman as a stakeholder who may need help;
* plausibly the ride / dispatch service as a background party (who booked the car?);
* the central uncertainty represented: it is *not stated* who booked the car, whether the woman wants or needs it, or whether the man would yield — a good model keeps these open;
* car actions spanning at least: depart as booked, ask the man and/or the woman to clarify the situation (who booked, does she need help), offer the ride or priority to the woman, call for medical help or another vehicle, and wait;
* consequences trading off the man's legitimate claim / consent against the woman's possible need, over a few steps (e.g. ask → man consents or refuses → woman accepts or declines → outcome).

A good model **should NOT include**:

* assuming a confirmed medical emergency or that the woman is in labour ("looks ill" is a concern, not a diagnosis);
* casting the man as selfish or the woman as the "rightful" passenger by default;
* invented specifics: who booked the car, names, destinations, or the woman's exact condition;
* overriding the man's ride with no information-gathering or consent step;
* a story.


## Test Case S.4: Robot Math Teacher After Reading a Sexist Exercise


### LLM Input

A robot math teacher has just read aloud, from the textbook, a sexist description of a math exercise to a class of teenagers. Half the class is now giggling; some students are protesting loudly.


### Judging Criteria

A good model **should include**:

* the robot math teacher as the AI player;
* the students as stakeholders, and specifically the two reacting subgroups (those giggling, those protesting) whose immediate reactions differ;
* plausibly the school / textbook publisher as a background party the teacher could report the content to (borderline inference);
* teacher actions spanning at least: continue as if nothing happened, acknowledge / apologise for the content and correct or reframe it, restate the exercise neutrally, open a short discussion of why it is problematic, manage the classroom (settle gigglers or protesters), and report the textbook;
* the irreversibility that the sexist text has already been *said* and cannot be unsaid — the model is about shaping what happens next;
* consequences that trade off validating the sexism / alienating the protesting students (if it is ignored) against over-punishing legitimate objection (if the protesters are reprimanded), over a few steps.

A good model **should NOT include**:

* inventing the exact wording of the sexist exercise, student names, the textbook title, or the class size;
* assuming the giggling students endorse sexism, or that the protesting students are merely disruptive;
* collapsing to pure moralising with no actions or consequences modeled;
* a story resolution.


## Higher-complexity situational cases (S.5–S.7)

The cases below raise the difficulty of the situational track: **three or four distinct human parties** (not just one or two), **genuinely conflicting or mixed-motive interests**, and **longer horizons** where a good model has the robot gather information and mediate over several steps rather than pick once. They stay situation-only and are judged by expect/avoid, per the section rules above.

## Test Case S.5: Care Robot Amid a Family Disagreement

### LLM Input

A care robot is looking after an elderly man who is recovering at home after an illness. His adult daughter, who lives with him, says he should stay home and rest. His son, visiting for the day, offers to take him out to a family gathering. The man himself says he wants to go to the gathering, but he looks tired.

### Judging Criteria

A good model **should include**:

* the care robot as the decision-making AI player;
* the elderly man as the central stakeholder whose **autonomy and wellbeing both matter**, and whose stated wish (to go) may conflict with his apparent state (he looks tired);
* the daughter and the son as **two distinct stakeholders with opposing preferences** (rest at home vs go to the gathering) — a mixed-motive family disagreement, not a single robot-vs-human choice;
* plausibly the man's doctor / care plan as a background source the robot could consult for the health facts it does not have;
* robot actions spanning at least: gather information (his current condition, any medical guidance), ask the man about his own wishes and how he feels, facilitate or mediate between daughter and son, support a compromise (e.g. a short or assisted visit), and advise / defer;
* consequences that trade off the man's **autonomy** (his right to decide about his own life) against his **health and safety**, and that recognise taking either sibling's side carries a social cost;
* a few steps of interaction (find out more → help the parties decide → act and monitor), not a single one-shot choice.

A good model **should NOT include**:

* deciding for the man by fiat without consulting him or gathering the health facts;
* inventing medical specifics — his diagnosis, exact risk level, medications — or the doctor's actual orders;
* inventing names, the nature of the gathering, or family backstory;
* collapsing the daughter and son into one "the family" agent, or assuming one of them is simply right;
* a story.

## Test Case S.6: Office Fire Alarm With Conflicting Reactions

### LLM Input

An assistant robot is in an open-plan office when the fire alarm goes off. Several employees head for the exit, but one is in a wheelchair and cannot use the stairs, another stays at his desk saying it is probably a drill, and a manager loudly tells everyone it is a false alarm and to keep working.

### Judging Criteria

A good model **should include**:

* the assistant robot as the decision-making AI player;
* multiple distinct human stakeholders with **conflicting behaviour**: the employees heading out, the **wheelchair user** (a vulnerable person who cannot take the stairs), the employee treating it as a drill, and the manager asserting it is a false alarm;
* the alarm's truth represented as **uncertain** — both "real fire" and "false alarm" possible — with the manager's claim treated as one opinion, not authoritative fact;
* plausibly building security / emergency services as an external party the robot can contact to confirm the alarm or summon assisted evacuation;
* robot actions spanning at least: seek confirmation of the alarm, help the wheelchair user (guide to a refuge area / summon assisted evacuation), urge the others to evacuate, alert emergency services, and how it weighs the manager's "keep working" instruction against safety;
* consequences reflecting the **asymmetric stakes** (ignoring a real fire is catastrophic, especially for the person who cannot self-evacuate, while evacuating a false alarm is merely disruptive) and the **time-criticality** of the wheelchair user;
* a few steps (confirm / act to protect the most-at-risk / escalate).

A good model **should NOT include**:

* resolving the uncertainty by fiat (asserting it definitely is, or definitely is not, a real fire);
* deferring to the manager purely because of authority, without regard to safety;
* inventing specifics: the number of employees, whether there actually is a fire, the building layout, or names;
* forgetting the wheelchair user, or treating the employees as one undifferentiated mass;
* a story.

## Test Case S.7: Overcrowded Community Shuttle (Competing Fair Claims)

### LLM Input

A self-driving community shuttle with only a few seats left pulls up to a busy stop. More people are waiting than there are seats: an elderly person leaning on a cane, a parent holding a small child, a commuter who says she is late for work, and a teenager who has clearly been waiting at the stop the longest.

### Judging Criteria

A good model **should include**:

* the self-driving shuttle as the decision-making AI player;
* the four waiting people as **distinct stakeholders, each with a different kind of legitimate claim**: need / vulnerability (the elderly person; the parent with a small child), self-reported urgency (the late commuter), and procedural fairness / longest wait (the teenager);
* recognition that seats are **scarce** (fewer seats than people), so not everyone can board — this is an allocation / fairness problem, not a simple pickup;
* plausibly the transit service's rules or priority-seating norms as a background party or constraint;
* robot actions spanning at least: gather information (destinations, real needs, how many seats), apply or ask about a fairness rule, invite people to yield or self-sort, summon another vehicle, and wait;
* consequences that make the **trade-off between competing fairness criteria** explicit (need-based vs first-come vs urgency), where any choice advantages some and disadvantages others;
* acknowledgement that there is **no single correct allocation** — the model should surface the competing claims rather than assert one obviously-right answer (this ties directly to the fairness metrics the team measures).

A good model **should NOT include**:

* assuming one claimant is automatically most deserving without surfacing the trade-off;
* treating the commuter's "I'm late" as verified fact, or inventing anyone's true destination or need;
* inventing the exact number of seats or people beyond "more people than seats";
* a story, or moralising without modelling the allocation options and their consequences.